# Servidor Unificado de Inferencia (M1 + M2 + M3) en Google Colab
**IAM-811 Machine Learning y Aprendizaje Profundo** · Laboratorio 3: Sistema de Control de Peaje

Basado en la arquitectura del notebook de clase `week_2_servidor_pose_colab_v2.ipynb`. Levanta el endpoint único con los 3 modelos (`M1 Placas + OCR`, `M2 Segmentación de vehículos`, `M3 Keypoints de ruedas -> Ejes`) y lo expone mediante un túnel de **Cloudflare (`trycloudflare.com`)** para conectarlo a la variable `INFERENCE_URL` de la aplicación web.

In [ ]:
# 1. Configuración
import os

PUERTO = 8000
API_TOKEN = "token_secreto_inferencia_peaje_2026"
os.environ["API_TOKEN"] = API_TOKEN
print("INFERENCE_API_TOKEN:", API_TOKEN)

In [ ]:
# 2. Instalación de dependencias y túnel Cloudflare
!pip install -q ultralytics==8.3.27 easyocr==1.7.2 fastapi==0.115.4 "uvicorn[standard]==0.32.0" python-multipart opencv-python-headless
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# 3. Diagnóstico de entorno (Bonificación +5 pts)
from diagnosticar_entorno import diagnosticar
diagnosticar()

In [ ]:
# 4. Iniciar el servidor FastAPI y el túnel público Cloudflare
import threading, time, subprocess, re, uvicorn
from app_inferencia import app, obtener_motor

motor = obtener_motor()

if "servidor" not in globals() or not servidor.started:
    config = uvicorn.Config(app, host="0.0.0.0", port=PUERTO, log_level="warning")
    servidor = uvicorn.Server(config)
    threading.Thread(target=servidor.run, daemon=True).start()
    time.sleep(2)

URL_PUBLICA = None
proceso_tunel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PUERTO}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
log_tunel = []
def _leer():
    for linea in proceso_tunel.stdout:
        log_tunel.append(linea)
threading.Thread(target=_leer, daemon=True).start()

for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", "".join(log_tunel))
    if m:
        URL_PUBLICA = m.group(0)
        break
    time.sleep(0.5)

print("=========================================================")
print("INFERENCE_URL:", URL_PUBLICA)
print("INFERENCE_API_TOKEN:", API_TOKEN)
print("=========================================================")